# Fundamentals 11 - Multi-Agentic Graph API

Graph multi-agente con nodos deterministas y un nodo LM opcional. El graph coordina estado; `runtime(provider="auto")` decide backend para el agente LM.


In [ ]:
import importlib.util
from typing import TypedDict
import os
import agentic_systems as lab
PRETTY = False
scheduler = lab.scheduler(timeout_s=60, max_retries=0, max_tool_calls=8, max_turns=8)
local_runtime = lab.runtime(provider="python-direct", model="local-python", region="local", scheduler=scheduler)
lm_runtime = lab.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
workspace = lab.AgenticSystem(model=lm_runtime.model_id or "local-python", region=lm_runtime.region_name or "local", runtime=lm_runtime)
lab.show({"local_runtime": local_runtime.describe(), "lm_runtime": lm_resolution, "lm_available": lm_available, "force_local_only": force_local_only})


## 1) Tools, agentes y estado


In [ ]:
@lab.tool
def record_review(summary: str) -> dict:
    """Registra una revisi?n LM como evidencia estructurada."""
    return {"summary": summary}

USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42

@lab.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b; x2 = x1 - c; x3 = x2 * d; x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}

@lab.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}

policy = lab.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
solver = lab.agent(name="graph_solver", instructions="Resuelve n?meros estructurados.", tools=[solve_arithmetic], engine="python-direct", runtime=local_runtime, contract=lab.AgentContract(must_call=["solve_arithmetic"], tool_expectation=lab.expect.exactly("solve_arithmetic")), policy=policy)
judge = lab.agent(name="graph_judge", instructions="Valida resultado.", tools=[judge_result], engine="python-direct", runtime=local_runtime, contract=lab.AgentContract(must_call=["judge_result"], tool_expectation=lab.expect.exactly("judge_result")), policy=policy)
reviewer = workspace.agent(name="graph_lm_reviewer", instructions="Resume riesgos sin cambiar el resultado.", tools=[record_review], runtime=lm_runtime, policy=lab.RunPolicy.for_mode("eval"))

class GraphState(TypedDict, total=False):
    prompt: str
    numbers: list[int]
    procedure: list[str]
    result: float
    judge: dict
    lm_review: str | None

lab.show({"agents": [solver.info(), judge.info(), reviewer.info()]})


## 2) Nodos del graph


In [ ]:
def solve_node(state: GraphState) -> GraphState:
    result = solver.run({"tool": "solve_arithmetic", "input": {"numbers": state["numbers"]}})
    return {**state, "procedure": result.data["procedure"], "result": result.data["result"], "solve_result": result}

def judge_node(state: GraphState) -> GraphState:
    result = judge.run({"tool": "judge_result", "input": {"result": state["result"], "expected": EXPECTED}})
    return {**state, "judge": result.data, "judge_result": result}

def review_node(state: GraphState) -> GraphState:
    if not lm_available:
        return {**state, "lm_review": None, "review_result": None}
    result = reviewer.run(str({"procedure": state["procedure"], "result": state["result"], "judge": state["judge"]}))
    return {**state, "lm_review": result.text, "review_result": result}

nodes = {"solve": solve_node, "judge": judge_node, "review": review_node}
edges = [("START", "solve"), ("solve", "judge"), ("judge", "review"), ("review", "END")]
lab.show({"nodes": list(nodes), "edges": edges})

## 3) Ejecutar con LangGraph si esta disponible; fallback local si no


In [ ]:
initial_state: GraphState = {"prompt": USER_PROMPT, "numbers": NUMBERS}
if importlib.util.find_spec("langgraph"):
    graph = lab.graph(state=GraphState, nodes=nodes, edges=edges, engine="langgraph", name="fundamentals_multi_agentic_graph")
    final_state = graph.run(initial_state)
    framework = "langgraph"
else:
    final_state = initial_state
    for node in [solve_node, judge_node, review_node]:
        final_state = node(final_state)
    framework = "local-state-pipeline"

payload = {
    "procedimiento": final_state["procedure"],
    "resultado_final": final_state["result"],
    "judge": final_state["judge"],
    "lm_review": final_state.get("lm_review"),
}
result = lab.compose_result(
    text="Graph multi-agente ejecutado.",
    data=payload,
    results=[final_state.get("solve_result"), final_state.get("judge_result"), final_state.get("review_result")],
    mode="multi-agentic-graph",
    framework=framework,
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution},
)
lineage = result.lineage(name="fundamentals.multi_agentic_graph", question=USER_PROMPT, goal="Explicar nodos, edges y estado final.")
lab.show({"framework": framework, "final_state_keys": list(final_state)})
lab.human_result(result, title="Human result - Multi-Agentic Graph", pretty=PRETTY, show_lineage=True, lineage=lineage)

## Coverage API


In [ ]:
lab.show({"notebook": "11_multi-agentic-graph-api.ipynb", "api_coverage": ["lab.graph", "nodes", "edges", "runtime(provider='auto')", "python-direct nodes", "compose_result", "RunResult.lineage"]})
